# 10 — Explicit Render (Pull Model)

In pull model, render is explicit — no auto-render on data change. Batching is natural.

**What you learn:**
- Data changes do NOT auto-render
- Call `render()` explicitly when you want output
- Multiple changes + one `render()` = efficient batching
- No need for suspend/resume

**Prerequisites:** 08_reactive_basics

In [ ]:
from pathlib import Path
from IPython.display import HTML

from genro_bag import Bag
from genro_bag.resolvers import FileResolver
from genro_builders.contrib.html import HtmlBuilder
from genro_builders.manager import ReactiveManager
from genro_toolbox import set_sync

set_sync()

In [ ]:
render_count = [0]
original_render = HtmlBuilder.render

class CountingBuilder(HtmlBuilder):
    def render(self, *args, **kwargs):
        render_count[0] += 1
        return original_render(self, *args, **kwargs)

class Dashboard(ReactiveManager):
    def on_init(self):
        self.page = self.register_builder('page', CountingBuilder)
        self.run(subscribe=True)

    def main(self, source):
        body = source.body()
        body.h1('Weather Dashboard')
        body.p('^temperature')
        body.p('^humidity')
        body.p('^pressure')
        body.p('^wind_speed')

app = Dashboard()
app.local_store('page').fill_from(
    Bag.from_json(Path('data.json').read_text()),
)
app.page.build()
store = app.local_store('page')
render_count[0] = 0

## Pull model: no auto-render

Data changes do not trigger render. Call `render()` when ready.

In [ ]:
store['temperature'] = 25
store['humidity'] = 60
store['pressure'] = 1015
store['wind_speed'] = 8
print(f'4 data changes, renders so far: {render_count[0]}')
print('(No auto-render in pull model)')

output = app.page.render()
print(f'After explicit render(): {render_count[0]} render total')
HTML(output)

## Natural batching

Change multiple values, render once. No suspend/resume needed.

In [ ]:
render_count[0] = 0

store['temperature'] = 30
store['humidity'] = 75
store['pressure'] = 1010
store['wind_speed'] = 15

output = app.page.render()
print(f'4 changes + 1 explicit render = {render_count[0]} render call')
HTML(output)